# Suite VLGDEF — Verilog definitions

`Definitions.define(unit)` gives the entities a source text declares, in its scopes, as IEEE 1800 resolves
names; `referents` tells what a name refers to.

In [ ]:
import { readFileSync } from "node:fs";

import { Syntax as F } from "@mbse/programs/Framework";
import { Definitions as D, SystemVerilog2023, Syntax as S } from "@mbse/programs/Verilog";

import { assert, equal } from "./support.js";

const SOURCES = "../../conformance/sources";
function define(text: string): [S.SourceText, ReturnType<typeof D.define>] {
  const unit = SystemVerilog2023.parse(text);
  return [unit, D.define(unit)];
}
const names = (unit: S.SourceText, spelling: string) => [...F.walk(unit)].filter((n) => n instanceof S.NameExpression
  && n.name instanceof S.Identifier && n.name.spelling === spelling);
const show = (entities: Iterable<unknown>) => [...entities].map(String);
const name = (spelling: string) => new S.Identifier({ spelling });
let unit: S.SourceText, program: ReturnType<typeof D.define>, entities: string[];

## VLGDEF-01 · Every entity of the conformance source, with its kind and qualified name

In [ ]:
unit = SystemVerilog2023.parse(readFileSync(`${SOURCES}/systemverilog2023.sv`, "utf8"), [SOURCES]);
program = D.define(unit);
entities = show(program.entities());
equal(entities.slice(0, 6), ["<package logger_pkg>", "<parameter logger_pkg::FIELDS>", "<localparam logger_pkg::BATTERY_LOW>",
  "<enumerator logger_pkg::IDLE>", "<enumerator logger_pkg::SAMPLE>", "<enumerator logger_pkg::SEND>"]);
assert(entities.includes("<argument logger_pkg::in_range::raw>") && entities.includes("<block sampler.counter>"));
assert(entities.includes("<localparam sampler.four.HALF>") && entities.includes("<instance sampler.u_fifo>"));
const unresolved = [...new Set([...F.walk(unit)].filter((n) => n instanceof S.NameExpression && n.name instanceof S.Identifier
  && D.referents(program, n).length === 0).map((n: any) => n.name.spelling))].sort();
equal(unresolved, ["option", "tb"]); // a covergroup's built-in `option`, and an outside module
console.log(entities.length, "entities");

## VLGDEF-02 · Imports: explicit ones alias, wildcard ones are found where nothing nearer is

In [ ]:
[unit, program] = define(`package p; parameter A = 1; parameter B = 2; parameter C = 3; endpackage
package q; parameter C = 4; endpackage
module m; import p::A; import q::*; import p::missing; import nowhere::*; import nowhere::Y;
parameter B = 5;
initial x = A + B + C + p::B + nowhere::X + p::B2;
endmodule`);
equal(show(D.referents(program, names(unit, "A")[0])), ["<parameter p::A>"]);
equal(show(D.referents(program, names(unit, "B")[0])), ["<parameter m.B>"]);
equal(show(D.referents(program, names(unit, "C")[0])), ["<parameter q::C>"]);
const scoped = [...F.walk(unit)].filter((n) => n instanceof S.ScopedName);
equal(show(D.referents(program, scoped[0])), ["<parameter p::B>"]);
assert(D.referents(program, scoped[1]).length === 0 && D.referents(program, scoped[2]).length === 0);
const missing = [...program.entities()].filter((e) => e.name === "missing")[0];
assert(missing !== undefined && missing.kind === "import" && missing.target === null);
console.log("ok");

## VLGDEF-03 · Ports, parameters, types, enumerators, functions and blocks, each in its scope

In [ ]:
[unit, program] = define(`module m #(parameter type T = int, parameter N) (a, b);
input a; output b; typedef enum {X, Y} e_t;
genvar g;
function int f(int v); begin : inner int t; t = v; end return v; endfunction
initial begin int w; for (int i = 0; i < 2; i++) w = i; foreach (q[j]) w = j; for (;;) ; end
for (g = 0; g < 2; g++) begin : gen end
endmodule`);
entities = show(program.entities());
equal(entities.slice(0, 7), ["<module m>", "<type parameter m.T>", "<parameter m.N>", "<port m.a>", "<port m.b>",
  "<enumerator m.X>", "<enumerator m.Y>"]);
const port = [...program.entities()].filter((e) => e.name === "a")[0];
assert(port !== undefined && port.declarations.length === 2 && port.definition instanceof S.PortReference);
for (const entity of ["<function m.f>", "<argument m.f.v>", "<block m.f.inner>", "<variable m.f.inner.t>", "<block m.gen>",
  "<genvar m.g>"]) {
  assert(entities.includes(entity), entity);
}
equal(D.referents(program, names(unit, "i")[0]).map((e) => e.kind), ["variable"]);
equal(D.referents(program, names(unit, "j")[0]).map((e) => e.kind), ["variable"]);
equal(D.referents(program, names(unit, "q")[0]), []);
const declared = [...F.walk(unit)].filter((n) => n instanceof S.Identifier && n.spelling === "w")[0];
equal(show(D.referents(program, declared)), ["<variable w>"]);
equal(D.referents(program, new S.Identifier({ spelling: "z" })), []);
console.log("ok");

## VLGDEF-04 · Interfaces, modports, instances and out-of-block names

In [ ]:
[, program] = define(`interface bus; logic d; modport mp (input d); endinterface
module top; bus b(); sub u (.p(b.d)); endmodule
function int c::f(); endfunction`);
entities = show(program.entities());
assert(entities.includes("<modport bus.mp>") && entities.includes("<instance top.b>") && entities.includes("<instance top.u>"));
assert(![...program.entities()].some((e) => e.kind === "function" && e.name === "f"));
console.log("ok");

## VLGDEF-05 · Classes: members, bases, constructors and methods defined outside their class

In [ ]:
[unit, program] = define(`class base; int x; extern function void f(int a); function new(); endfunction endclass
class derived extends base #(1); function void g(); y = x; endfunction endclass
interface class i1; parameter int P = 1; endclass
interface class i2 extends i1; endclass
function void base::f(int a); x = a + i2::P; endfunction
function void nowhere::h(); endfunction
function void base::missing(); endfunction
package p; class c; static int s; endclass endpackage
class from_package extends p::c; endclass
class from_nowhere extends unknown; endclass
class reader; function void r(); y = p::c::s + p::q::s + q::r::s + p::c::missing::s; endfunction endclass
`);
entities = show(program.entities());
equal(entities.slice(0, 6), ["<class base>", "<variable base::x>", "<function base::f>", "<argument base::f::a>",
  "<function base::new>", "<class derived>"]);
const f = [...program.entities()].filter((e) => e.name === "f")[0] as any;
assert(f.declarations.length === 2 && f.definition === unit.items[4] && f.scope.parent.owner.name === "base");
equal(show(D.referents(program, names(unit, "x")[0])), ["<variable base::x>"]); // through derived's base
equal(show(D.referents(program, names(unit, "x")[1])), ["<variable base::x>"]);
equal(show(D.referents(program, names(unit, "a")[0])), ["<argument base::f::a>"]);
equal(D.referents(program, names(unit, "y")[0]), []);
const scopedNames = [...F.walk(unit)].filter((n) => n instanceof S.ScopedName) as S.ScopedName[];
equal(show(D.referents(program, scopedNames[1])), ["<parameter i1::P>"]); // through i2's interfaces
equal(show(D.referents(program, scopedNames[0])), ["<function base::f>"]);
assert(D.referents(program, scopedNames[2]).length === 0 && D.referents(program, scopedNames[3]).length === 0); // nowhere::h, base::missing
const nested = scopedNames.filter((n) => n.name instanceof S.ScopedName);
equal(show(D.referents(program, nested[0])), ["<variable p::c::s>"]);
equal(nested.slice(1).map((n) => D.referents(program, n)), [[], [], [], []]); // no q in p, no q, no missing in c (twice)
const classes = new Map([...program.entities()].filter((e) => e.kind === "class").map((e) => [e.name, e]));
equal(classes.get("from_package")?.scope?.bases, [classes.get("c")?.scope]);
equal(classes.get("from_nowhere")?.scope?.bases, []);
const handmade = new S.SourceText({ items: [new S.FunctionDeclaration({ name: new S.ScopedName({ scope: name("p"),
  name: new S.ScopedName({ scope: name("c"), name: name("f") }) }) })] });
equal([...D.define(handmade).entities()], []); // a qualified name slang does not read: not resolved
equal(show(D.referents(program, new S.ScopedName({ scope: name("base"), name: name("x") }))), ["<variable base::x>"]);
console.log("ok");

## VLGDEF-06 · Constraints, their loops, `local::` and array methods' iterators

In [ ]:
[unit, program] = define(`class c; rand int x, y; int q[]; extern constraint e; pure constraint p;
constraint k { foreach (q[i]) q[i] < x; }
function void f(int y); ok = randomize() with { x < local::y; }; r = q.sum() with (item * 2); r = q.find(v) with (v > y); q.sort with (item.x);
endfunction endclass
constraint c::e { x != y; }
constraint nowhere::e { }
`);
entities = show(program.entities());
equal(entities.slice(0, 7), ["<class c>", "<variable c::x>", "<variable c::y>", "<variable c::q>", "<constraint c::e>",
  "<constraint c::p>", "<constraint c::k>"]);
const outside = [...program.entities()].filter((e) => e.name === "e")[0] as any;
assert(outside.declarations.length === 2 && outside.definition === unit.items[1]);
equal(show(D.referents(program, names(unit, "i")[0])), ["<variable i>"]);
const local = [...F.walk(unit)].filter((n) => n instanceof S.NameExpression && n.name instanceof S.LocalName)[0];
equal(show(D.referents(program, local)), ["<argument c::f::y>"]);
equal(show(D.referents(program, names(unit, "item")[0])), ["<variable item>"]);
equal(show(D.referents(program, names(unit, "item")[1])), ["<variable item>"]); // q.sort's
equal(show(D.referents(program, names(unit, "v")[0])), ["<variable v>"]);
equal(show(D.referents(program, names(unit, "y").at(-1))), ["<variable c::y>"]); // in c::e, the class's
console.log("ok");

## VLGDEF-07 · Properties, sequences, `let`s, clocking blocks and labels

In [ ]:
[unit, program] = define(`module m; logic clk, a, b, rst;
default clocking cb @(posedge clk); input #1 a; output b2 = b; endclocking
default clocking @(negedge clk); input a; endclocking
sequence s(x, int y = 1); int v; (a, v = x) ##1 b ##(y) 1; endsequence
property p(sequence q); disable iff (rst) q |=> a; endproperty
let twice(n) = n * 2;
chk: assert property (@(posedge clk) p(s(a)));
initial begin lab: a = twice(1); cb.b2 <= 1; end
default clocking cb;
endmodule`);
entities = show(program.entities());
for (const entity of ["<clocking m.cb>", "<clockvar m.cb.a>", "<clockvar m.cb.b2>", "<sequence m.s>", "<argument m.s.x>",
  "<argument m.s.y>", "<variable m.s.v>", "<property m.p>", "<argument m.p.q>", "<let m.twice>", "<argument m.twice.n>",
  "<label m.chk>", "<label lab>"]) {
  assert(entities.includes(entity), entity);
}
equal(show(D.referents(program, names(unit, "x")[0])), ["<argument m.s.x>"]);
equal(show(D.referents(program, names(unit, "y")[0])), ["<argument m.s.y>"]);
equal(show(D.referents(program, names(unit, "v")[0])), ["<variable m.s.v>"]);
equal(show(D.referents(program, names(unit, "q")[0])), ["<argument m.p.q>"]);
equal(show(D.referents(program, names(unit, "n")[0])), ["<argument m.twice.n>"]);
equal(show(D.referents(program, names(unit, "b")[0])), ["<variable m.b>"]); // a clockvar's value is outside
equal(show(D.referents(program, names(unit, "twice")[0])), ["<let m.twice>"]);
const reference = [...F.walk(unit)].filter((n) => n instanceof S.DefaultClocking)[0] as S.DefaultClocking;
equal(show(D.referents(program, reference.name)), ["<clocking m.cb>"]);
console.log("ok");

## VLGDEF-08 · Covergroups, their arguments, coverpoints, crosses, bins and filters

In [ ]:
[unit, program] = define(`class c; int a, b, en;
covergroup cg(int lo = 0, hi) @(posedge en);
  cp_a: coverpoint a iff (en) { bins low = {[0:lo]} with (item > b) iff (en); bins s = cp_a with (item < 3); }
  coverpoint b { bins one = {1}; }
  x: cross cp_a, b { bins both = binsof(cp_a.low) && binsof(b); function int f(); return lo; endfunction }
endgroup
covergroup sampled with function sample(int v); coverpoint v; endgroup
endclass`);
entities = show(program.entities());
for (const entity of ["<covergroup c::cg>", "<argument c::cg::lo>", "<coverpoint c::cg::cp_a>", "<bins c::cg::cp_a::low>",
  "<bins c::cg::cp_a::s>", "<cross c::cg::x>", "<bins c::cg::x::both>", "<function c::cg::x::f>", "<covergroup c::sampled>",
  "<argument c::sampled::v>", "<bins one>"]) {
  assert(entities.includes(entity), entity);
}
equal(show(D.referents(program, names(unit, "lo")[0])), ["<argument c::cg::lo>"]);
equal(show(D.referents(program, names(unit, "item")[0])), ["<variable item>"]);
equal(show(D.referents(program, names(unit, "b")[1])), ["<variable c::b>"]); // in the filter
equal(show(D.referents(program, names(unit, "cp_a")[0])), ["<coverpoint c::cg::cp_a>"]);
equal(show(D.referents(program, names(unit, "lo").at(-1))), ["<argument c::cg::lo>"]); // in x's function
equal(show(D.referents(program, names(unit, "v")[0])), ["<argument c::sampled::v>"]);
assert(entities.includes("<argument c::cg::hi>"));
equal(show(D.referents(program, names(unit, "en").at(-1))), ["<variable c::en>"]); // a bin's condition
console.log("ok");